In [1]:
# Run this in a new cell then restart kernel
!pip install --upgrade numpy scipy adjustText
!pip install "numpy>=2.0.0,<2.4.0" "scipy>=1.14.0" "adjustText>=1.0.0"

  Using cached numpy-2.3.5-cp312-cp312-win_amd64.whl.metadata (60 kB)
Using cached numpy-2.3.5-cp312-cp312-win_amd64.whl (12.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.4
    Uninstalling numpy-2.5.4:
      Successfully uninstalled numpy-2.5.4


  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.3.5 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.18.1 which is incompatible.
mlxtend 0.24.0 requires pandas>=2.3.3, but you have pandas 2.2.3 which is incompatible.
mlxtend 0.24.0 requires scikit-learn>=1.8.0, but you have scikit-learn 1.6.1 which is incompatible.
streamlit 1.37.1 requires packaging<25,>=20, but you have packaging 26.3 which is incompatible.
streamlit 1.37.1 requires pillow<11,>=7.1.0, but you have pillow 12.3.0 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.33.6 whic

In [2]:
import joblib
import numpy as np
import pandas as pd
import ast
import pygsheets
import sklearn
assert sklearn.__version__ == "1.6.1", f"Expected sklearn 1.6.1, got {sklearn.__version__}"
from sklearn.preprocessing  import StandardScaler
from sklearn.impute         import SimpleImputer
from sklearn.pipeline       import Pipeline
from sklearn.compose        import ColumnTransformer
import warnings

In [3]:
!pip install "numpy<2"

  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.5
    Uninstalling numpy-2.3.5:
      Successfully uninstalled numpy-2.3.5


  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.18.1 which is incompatible.
jax 0.11.1 requires numpy>=2.1, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.11.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
mlxtend 0.24.0 requires numpy>=2.3.5, but you have numpy 1.26.4 which is incompatible.
mlxtend 0.24.0 requires pandas>=2.3.3, but you have pandas 2.2.3 which is incompatible.
mlxtend 0.24.0 requires scikit-learn>=1.8.0, but you have scikit-learn 1.6.1 which is incompatible.
ml-dtypes 0.6.0 requires numpy>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-pyth

In [4]:
import numpy
print(numpy.__version__)

2.3.5


In [5]:
import sklearn
print(sklearn.__version__)   # should show 1.6.1 after upgrade

1.6.1


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — In your NEW notebook, load everything from the bundle
# ══════════════════════════════════════════════════════════════════════════════
bundle = joblib.load(r"C:\Users\Segun\Desktop\Model mangement\Classification model\14 days prediction model\Individual\Logisticv3id.pkl") #XGBadjv2.pkl #RandomForestv2.pkl #Logisticv2.pkl

loaded_model         = bundle["model"]
ALL_FEATURES         = bundle["all_features"]
NUMERIC_FEATURES     = bundle["numeric_features"]
CATEGORICAL_FEATURES = bundle["categorical_features"]

print(f"Model type   : {type(loaded_model)}")
print(f"All features : {len(ALL_FEATURES)}")
assert hasattr(loaded_model, "predict_proba"), "Not a valid model"
print("Ready to predict ✓")

Model type   : <class 'sklearn.pipeline.Pipeline'>
All features : 314
Ready to predict ✓


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — Load new signals and encode (same as training notebook)
# ══════════════════════════════════════════════════════════════════════════════
SERVICE_ACCOUNT_FILE  = r"C:\Users\Segun\Desktop\wqu-satellite-image-learning1-175e0a0906aa.json"
SHEET_URL             = "https://docs.google.com/spreadsheets/d/1jkbrrHnR48JztVz0SrqZELKjOenVm44wsiIchLW0K9g/edit#gid=1868316212"
SAMPLE_WORKSHEET_INDEX = 2     # ← your unseen signals tab
TIME_COL               = "time"
TICKER_COL             = "name"
SAMPLE_TEST_START      = "2026-10-15"
SAMPLE_TEST_END        = "2026-10-20"

# Load sheet
gc       = pygsheets.authorize(service_account_file=SERVICE_ACCOUNT_FILE)
sh       = gc.open_by_url(SHEET_URL)
wks      = sh[SAMPLE_WORKSHEET_INDEX]
df_sample = wks.get_as_df()

# Clean
df_sample = df_sample.loc[:, ~df_sample.columns.duplicated(keep="first")]
df_sample = df_sample[[c for c in df_sample.columns if str(c).strip() != ""]]
df_sample = df_sample.replace("", np.nan).dropna(how="all").reset_index(drop=True)

# Parse datetime
df_sample[TIME_COL] = (
    pd.to_datetime(df_sample[TIME_COL], utc=True, errors="coerce")
      .dt.tz_convert("America/New_York")
      .dt.tz_localize(None)
)

# Coerce numerics
skip_text = {TIME_COL, TICKER_COL, "sector", "industry", "BreakoutPatterns"}
for col in df_sample.columns:
    if col not in skip_text and isinstance(df_sample[col], pd.Series):
        converted = pd.to_numeric(df_sample[col], errors="coerce")
        if converted.notna().sum() > 0:
            df_sample[col] = converted

df_sample = df_sample.sort_values(TIME_COL).reset_index(drop=True)

C:\Users\Segun\AppData\Local\Temp\ipykernel_31320\3146247719.py:21: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_sample = df_sample.replace("", np.nan).dropna(how="all").reset_index(drop=True)


In [8]:
df_sample

,name,BreakoutPatterns,PatternScore,sector,industry,close,time,VWAP,VWAP|1,VWAP|5,...,relative_volume,relative_volume_intraday|5,MoneyFlow,MoneyFlow|1,MoneyFlow|5,MoneyFlow|15,MoneyFlow|30,MoneyFlow|1W,MoneyFlow|1M,vix_close
0,IOR,"['Candle.Marubozu.White|1W', 'Candle.LongShado...",36.5000,Finance,Real Estate Investment Trusts,17.8500,2026-05-26 09:30:00,17.533333,17.701485,17.679793,...,1.116402,2.612110,58.965759,53.646435,52.908091,52.908091,58.960735,64.515932,49.131231,NaN
1,GBLI,"['Candle.Doji.Dragonfly|1', 'Candle.Doji.Drago...",74.5000,Finance,Property/Casualty Insurance,26.9350,2026-05-26 09:30:00,26.928333,26.953378,26.950460,...,2.111177,4.720398,40.862909,51.053565,37.747328,21.753860,13.304465,52.660318,17.409826,NaN
2,IOR,"['Candle.Marubozu.White|1W', 'Candle.LongShado...",36.5000,Finance,Real Estate Investment Trusts,17.8500,2026-05-26 09:30:00,17.533333,17.701485,17.679793,...,1.116402,2.612110,58.965759,53.646435,52.908091,52.908091,58.960735,64.515932,49.131231,NaN
3,BTOC,"['Candle.Hammer|5', 'Candle.Doji.Dragonfly|1',...",74.6667,Transportation,Other Transportation,0.3035,2026-05-27 09:30:00,0.306867,0.305736,0.305676,...,0.290845,0.311432,66.499063,49.999254,13.281601,44.072855,42.349160,76.030644,9.206276,NaN
4,MATH,"['Candle.Marubozu.White|5', 'Candle.Doji.Drago...",49.6667,Transportation,Air Freight/Couriers,1.0800,2026-05-27 09:30:00,1.070000,1.076041,1.075657,...,0.515465,0.379158,33.512944,67.889061,17.508119,28.755739,49.866697,36.636532,62.797114,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78455,OSUR,"['Candle.Hammer|30', 'Candle.Hammer|60', 'Cand...",46.9800,Health Technology,Medical Specialties,3.6800,2026-10-19 09:30:00,3.656667,3.664640,3.664469,...,1.332726,0.786723,60.478705,72.893578,88.843087,72.725038,69.824551,45.267381,61.017393,19.06
78456,TMC,"['Candle.Hammer', 'Candle.Hammer|1W']",46.9800,Non-Energy Minerals,Other Metals/Minerals,2.6200,2026-10-19 09:30:00,2.606667,2.620173,2.620097,...,0.575211,0.541278,18.811399,3.134833,34.723315,58.831593,71.508013,44.102261,67.866375,19.06
78457,LTBR,"['Candle.Hammer', 'Candle.Hammer|1W']",46.9800,Process Industries,Chemicals: Specialty,5.4100,2026-10-19 09:30:00,5.310000,5.260647,5.259166,...,0.893432,0.954882,39.618873,42.098474,72.383708,77.437528,72.705328,39.510744,58.806958,19.06
78458,BMRC,"['Candle.Piercing.Bullish|5', 'Candle.Spinning...",44.8200,Finance,Regional Banks,26.0700,2026-10-19 09:30:00,26.171667,26.160717,26.156895,...,0.816840,0.739398,33.060622,20.983644,47.644081,48.085007,21.452586,36.308352,56.865993,19.06


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# FILTER — narrow signals by sector or industry
# ══════════════════════════════════════════════════════════════════════════════
# ── What's available in this sheet? ──────────────────────────────────────────
print("Unique sectors:")
print(df_sample["sector"].value_counts().to_string())
print("\nUnique industries:")
print(df_sample["industry"].value_counts().to_string())

Unique sectors:
sector
Health Technology         14459
Finance                   13957
Technology Services        9675
Electronic Technology      5319
Producer Manufacturing     4597
Commercial Services        4046
Consumer Services          3045
Retail Trade               2686
Process Industries         2564
Transportation             2338
Consumer Non-Durables      2269
Industrial Services        2204
Consumer Durables          2056
Non-Energy Minerals        1841
Distribution Services      1639
Energy Minerals            1532
Health Services            1315
Miscellaneous              1210
Utilities                  1205
Communications              491

Unique industries:
industry
Packaged Software                     6094
Pharmaceuticals: Major                5551
Biotechnology                         4909
Regional Banks                        3624
Medical Specialties                   3533
Real Estate Investment Trusts         3210
Miscellaneous Commercial Services     2676
Industr

In [10]:
# ── Configure your filter here ────────────────────────────────────────────────
# Set to None to skip that filter level — both can be used together

FILTER_SECTORS    =  None #["Transportation"] # "Health Technology","Finance","Technology Services","Electronic Technology","Producer Manufacturing",   
                            #"Commercial Services","Consumer Services","Retail Trade","Process Industries","Transportation",            
                            #"Consumer Non-Durables","Industrial Services","Consumer Durables","Non-Energy Minerals",      
                            #"Distribution Services","Energy Minerals","Health Services","Miscellaneous","Utilities","Communications"  

FILTER_INDUSTRIES = None   # e.g. ["Pharmaceuticals: Major", "Packaged Software"]
                           # or None to include all industries within the sectors above

# ── Apply filter ──────────────────────────────────────────────────────────────
before = len(df_sample)

if FILTER_SECTORS is not None:
    df_sample = df_sample[df_sample["sector"].isin(FILTER_SECTORS)]

if FILTER_INDUSTRIES is not None:
    df_sample = df_sample[df_sample["industry"].isin(FILTER_INDUSTRIES)]

df_sample = df_sample.reset_index(drop=True)

print(f"\nFilter applied:")
if FILTER_SECTORS:
    print(f"  Sectors    : {FILTER_SECTORS}")
if FILTER_INDUSTRIES:
    print(f"  Industries : {FILTER_INDUSTRIES}")
print(f"\nSignals before filter : {before}")
print(f"Signals after filter  : {len(df_sample)}")
print(f"Removed               : {before - len(df_sample)}")

if len(df_sample) == 0:
    raise ValueError(
        "No signals remain after filtering — "
        "check FILTER_SECTORS / FILTER_INDUSTRIES match values printed above"
    )

print(f"\nRemaining sector breakdown:")
print(df_sample["sector"].value_counts().to_string())
print(f"\nRemaining industry breakdown:")
print(df_sample["industry"].value_counts().to_string())


Filter applied:

Signals before filter : 78460
Signals after filter  : 78460
Removed               : 0

Remaining sector breakdown:
sector
Health Technology         14459
Finance                   13957
Technology Services        9675
Electronic Technology      5319
Producer Manufacturing     4597
Commercial Services        4046
Consumer Services          3045
Retail Trade               2686
Process Industries         2564
Transportation             2338
Consumer Non-Durables      2269
Industrial Services        2204
Consumer Durables          2056
Non-Energy Minerals        1841
Distribution Services      1639
Energy Minerals            1532
Health Services            1315
Miscellaneous              1210
Utilities                  1205
Communications              491

Remaining industry breakdown:
industry
Packaged Software                     6094
Pharmaceuticals: Major                5551
Biotechnology                         4909
Regional Banks                        3624
Medical 

In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — Encode (must match training notebook exactly)
# ══════════════════════════════════════════════════════════════════════════════
def encode_patterns(df):
    df = df.copy()
    df["BreakoutPatterns"] = df["BreakoutPatterns"].fillna("[]")
    df["BreakoutPatterns"] = df["BreakoutPatterns"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else []
    )
    exploded = df[["BreakoutPatterns"]].explode("BreakoutPatterns")
    dummies  = pd.get_dummies(exploded["BreakoutPatterns"], prefix="pattern")
    dummies  = dummies.groupby(level=0).max()
    return pd.concat([df.drop(columns=["BreakoutPatterns"]), dummies], axis=1)

def encode_categoricals(df, cols, drop_first=False):
    existing = [c for c in cols if c in df.columns]
    return pd.get_dummies(df, columns=existing, drop_first=drop_first)

df_sample = encode_patterns(df_sample)
df_sample = encode_categoricals(df_sample, cols=["sector", "industry"], drop_first=False)

# Date filter
mask = (
    (df_sample[TIME_COL].dt.date >= pd.Timestamp(SAMPLE_TEST_START).date()) &
    (df_sample[TIME_COL].dt.date <= pd.Timestamp(SAMPLE_TEST_END).date())
)
df_sample = df_sample[mask].copy().reset_index(drop=True)
print(f"Signals found : {len(df_sample)}")

Signals found : 5029


In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 — Predict
# ══════════════════════════════════════════════════════════════════════════════
# reindex aligns columns to exactly what the model was trained on
X_sample = df_sample.reindex(columns=ALL_FEATURES, fill_value=0).copy()

# ── NEW: check for training features missing from this batch ──────────────────
missing = set(ALL_FEATURES) - set(df_sample.columns)
if missing:
    print(f"⚠️  {len(missing)} training features not present in this batch (filled with 0): {sorted(missing)[:10]}...")

bool_c   = X_sample.select_dtypes(include="bool").columns
X_sample[bool_c] = X_sample[bool_c].astype(int)
df_sample["pred_prob"]   = loaded_model.predict_proba(X_sample)[:, 1]
df_sample["pred_label"]  = (df_sample["pred_prob"] >= 0.80).astype(int)
df_sample["signal_date"] = df_sample[TIME_COL].dt.date
df_sample["rank"]        = df_sample.groupby("signal_date")["pred_prob"].rank(
                               ascending=False, method="first")

# ── Portfolio Top-N ────────────────────────────────────────────────────────────
TOP_N = 15
portfolio_sample = df_sample[df_sample["rank"] <= TOP_N].sort_values(["signal_date", "rank"])

print(f"\n{'='*60}")
print(f"  PORTFOLIO — Top-{TOP_N} signals per day Model")
print(f"  {SAMPLE_TEST_START} → {SAMPLE_TEST_END}")
print(f"{'='*60}")

for date, group in portfolio_sample.groupby("signal_date"):
    print(f"\n  📅 {date}")
    for _, r in group.iterrows():
        print(f"    #{int(r['rank'])}  {str(r[TICKER_COL]):6s}  "
              f"close=${float(r['close']):.4f}  "
              f"prob={r['pred_prob']:.3f}  "
              f"PatternScore={r.get('PatternScore', '?')}")


  PORTFOLIO — Top-15 signals per day Model
  2026-10-15 → 2026-10-20

  📅 2026-10-15
    #1  TGL     close=$0.3000  prob=1.000  PatternScore=36.8867
    #2  JBDI    close=$2.3400  prob=1.000  PatternScore=38.06
    #3  RPGL    close=$13.3300  prob=0.995  PatternScore=42.11
    #4  TCRT    close=$2.1800  prob=0.995  PatternScore=49.15
    #5  LNKS    close=$0.8399  prob=0.978  PatternScore=34.54
    #6  ABLV    close=$2.0900  prob=0.972  PatternScore=29.405
    #7  YMAT    close=$1.5100  prob=0.971  PatternScore=42.11
    #8  MNDR    close=$1.0900  prob=0.965  PatternScore=61.86
    #9  ZBAO    close=$0.9523  prob=0.963  PatternScore=42.7912
    #10  SPRC    close=$2.0000  prob=0.960  PatternScore=26.97
    #11  JYD     close=$1.3800  prob=0.936  PatternScore=74.03
    #12  ESLA    close=$0.5352  prob=0.933  PatternScore=38.06
    #13  PMI     close=$10.0000  prob=0.913  PatternScore=39.41
    #14  HCAI    close=$0.2792  prob=0.887  PatternScore=48.605
    #15  JAGX    close=$4.5900  p

In [13]:
portfolio_sample = portfolio_sample[['name','PatternScore','close','signal_date','rank','pred_prob','pred_label']]

In [14]:
# Authorize with service account
gc = pygsheets.authorize(service_account_file=r"C:\Users\Segun\Desktop\wqu-satellite-image-learning1-175e0a0906aa.json")

# Open the spreadsheet by URL
sh = gc.open_by_url("https://docs.google.com/spreadsheets/d/1jkbrrHnR48JztVz0SrqZELKjOenVm44wsiIchLW0K9g/edit#gid=1868316212")